# exp5 — temporal candidate recall audit

**Timestamp:** 2026-10-24 03:40 MSK  
**Run group:** `exp5-candidate-recall-tripartite`  
**Intent:** Raise candidate coverage before training any reranker. Candidate Recall@200/500 is a retrieval diagnostic. Candidate-stage MAP@5 is reported separately; no learned reranker or Kaggle score is claimed.

## Method and leakage controls

- Rich interaction source: `data/exp5_user_airings_rich.parquet`, created with every positive overlap and no early watch threshold.
- Primary split: history weeks 11–20 → target weeks 21–30. Rolling: ≤16→17–20, ≤20→21–24, ≤24→25–27, ≤27→28–30.
- Every candidate generator uses history strictly before its target window. Target schedule metadata is permitted, but the schedule is filtered by ISO week; `export_arh_21-30-final.csv` includes 267 week-31 rows, so those are excluded.
- Strong-watch relevant show: at least one target airing for that user has `user_watch_pct >= 0.8`; the 0.5 definition is reported as an ablation. Eligible shows must occur in the target-window schedule.
- Source C is the existing tripartite PPR helper (`exp3_e1_attr_ppr.py`), `attr_weight=0.5`, restart=0.5, 30 iterations. Early prototypes used bipartite PPR and an unfiltered schedule catalog; those results are preserved but superseded.

## Candidate sources

A: repeat history; B: LightFM WARP; C: tripartite PPR; D: recent popularity; E: user×channel×weekday×30-minute-slot schedule affinity. Expanded variants added longer lists, user×channel, user×metadata, ALS, user weekday×slot, channel×weekday/hour and recent popularity×future airings. Association rules were tested as an extra source.


## Recorded run commands

```bash
.venv-exp5/bin/python artifacts/scripts/exp5_candidate_recall.py baseline_corrected main
.venv-exp5/bin/python artifacts/scripts/exp5_candidate_recall.py expanded4 early
.venv-exp5/bin/python artifacts/scripts/exp5_candidate_recall.py expanded4 main
.venv-exp5/bin/python artifacts/scripts/exp5_candidate_recall.py expanded4 roll21_24
.venv-exp5/bin/python artifacts/scripts/exp5_candidate_recall.py expanded4 roll25_27
.venv-exp5/bin/python artifacts/scripts/exp5_candidate_recall.py expanded4 roll28_30
.venv/bin/python artifacts/scripts/exp5_tune_candidate_fusion.py tripartite
.venv-exp5/bin/python artifacts/scripts/exp5_candidate_recall.py expanded5 early
.venv-exp5/bin/python artifacts/scripts/exp5_candidate_recall.py expanded5 main
.venv-exp5/bin/python artifacts/scripts/exp5_candidate_recall.py expanded6 early
.venv-exp5/bin/python artifacts/scripts/exp5_candidate_recall.py expanded6 main
.venv/bin/python artifacts/scripts/exp5_tune_candidate_fusion.py expanded5
.venv/bin/python artifacts/scripts/exp5_tune_candidate_fusion.py expanded6
.venv/bin/python artifacts/scripts/exp5_eval_rolling_fusion.py
.venv/bin/python artifacts/scripts/exp5_tune_candidate_map.py
.venv/bin/python artifacts/scripts/exp5_tune_candidate_map.py widegrid
.venv/bin/python artifacts/scripts/exp5_tune_candidate_map.py propensity
.venv/bin/python artifacts/scripts/exp5_tune_candidate_map.py propensity_widegrid
.venv-exp5/bin/python artifacts/scripts/exp5_candidate_recall.py expanded7 early
.venv-exp5/bin/python artifacts/scripts/exp5_candidate_recall.py expanded7 main
```

The notebook code cells below read the actual saved metric/candidate artifacts and print their outputs captured at notebook creation. Every output table is under `results/tables/exp5_*.csv`; candidate rows are in ignored `data/exp5_*.parquet` files.


In [1]:
import pandas as pd
paths={
 'A-E tripartite baseline':'results/tables/exp5_candidate_recall_corrected.csv',
 'expanded4 tripartite':'results/tables/exp5_candidate_recall_expanded4.csv',
 'expanded5 time contexts':'results/tables/exp5_candidate_recall_expanded5.csv',
 'expanded6 + airing popularity':'results/tables/exp5_candidate_recall_expanded6.csv',
 'expanded7 + association rules':'results/tables/exp5_candidate_recall_expanded7.csv',
}
print('main strong-watch Recall@500 / full union')
for label,path in paths.items():
 d=pd.read_csv(path); r=d[(d.threshold==.8)&(d.generator=='rrf_union')]
 at500=r[r.cutoff==500].iloc[0]; full=r.iloc[-1]
 print(f'{label}: {at500.recall:.6f} / {full.recall:.6f} ({int(at500.hits)}/{int(at500.relevant)})')

main strong-watch Recall@500 / full union
A-E tripartite baseline: 0.494101 / 0.494101 (19643/39755)
expanded4 tripartite: 0.577386 / 0.837379 (22954/39755)
expanded5 time contexts: 0.609785 / 0.882983 (24242/39755)
expanded6 + airing popularity: 0.613961 / 0.887259 (24408/39755)
expanded7 + association rules: 0.608301 / 0.887687 (24183/39755)

In [2]:
roll=pd.read_csv('results/tables/exp5_candidate_fusion_rolling_tripartite.csv')
for split in ['roll21_24','roll25_27','roll28_30']:
 for ranking in ['equal_rrf','early_tuned']:
  r=roll[(roll.split==split)&(roll.ranking==ranking)&(roll.threshold==.8)&(roll.cutoff==500)].iloc[0]
  print(f'{split} {ranking}: recall@500={r.recall:.6f} ({int(r.hits)}/{int(r.relevant)}, users={int(r.users)})')
main=pd.read_csv('results/tables/exp5_candidate_fusion_tuned_tripartite.csv')
for ranking in ['equal_rrf','early_tuned']:
 r=main[(main.split=='main_holdout')&(main.ranking==ranking)&(main.threshold==.8)&(main.cutoff==500)].iloc[0]
 print(f'main {ranking}: recall@500={r.recall:.6f} ({int(r.hits)}/{int(r.relevant)}, users={int(r.users)})')

roll21_24 equal_rrf: recall@500=0.696085 (12107/17393, users=924)
roll21_24 early_tuned: recall@500=0.745530 (12967/17393, users=924)
roll25_27 equal_rrf: recall@500=0.816773 (5113/6260, users=982)
roll25_27 early_tuned: recall@500=0.850319 (5323/6260, users=982)
roll28_30 equal_rrf: recall@500=0.735432 (16117/21915, users=1208)
roll28_30 early_tuned: recall@500=0.793292 (17385/21915, users=1208)
main equal_rrf: recall@500=0.577361 (22953/39755, users=1236)
main early_tuned: recall@500=0.598692 (23801/39755, users=1236)

In [3]:
w=pd.read_csv('results/tables/exp5_candidate_fusion_weights_tripartite.csv')
print(w.to_string(index=False))
c=pd.read_parquet('data/exp5_candidates_main_expanded6.parquet')
print('candidate rows/users:',len(c),c.user_id.nunique())
print('mean candidates/user:',round(c.groupby('user_id').size().mean(),1))
print('source flags:',', '.join(col[3:] for col in c if col.startswith('in_')))
print('collaborative signals:',', '.join(col for col in ['lightfm_score','lightfm_rank','user_bias','item_bias','ppr_score','ppr_rank','als_score','als_rank'] if col in c))

     source  weight              tuning_split
     repeat     0.0 history<=16,target=17-20
    lightfm     0.0 history<=16,target=17-20
        ppr     0.0 history<=16,target=17-20
 popularity     1.0 history<=16,target=17-20
    tv_slot     1.0 history<=16,target=17-20
user_channel     4.0 history<=16,target=17-20
user_metadata     0.0 history<=16,target=17-20
        als     1.0 history<=16,target=17-20
candidate rows/users: 2585877 1244
mean candidates/user: 2078.7
source flags: repeat, lightfm, ppr, popularity, tv_slot, user_channel, user_time_slot, user_channel_dow, user_channel_hour, airing_popularity, user_metadata, als
collaborative signals: lightfm_score, lightfm_rank, user_bias, item_bias, ppr_score, ppr_rank, als_score, als_rank

## Failed attempts and resolutions

- First PPR normalization attempt built a rectangular diagonal and failed with `ValueError: axis 1 index 4840 exceeds matrix dimension 2973`; replaced it with `scipy.sparse.diags`. The later graph-type review found that the first successful candidate experiments still used bipartite PPR, not the requested tripartite PPR. Those tables remain, explicitly superseded.
- First ALS run fit an item×user matrix but indexed it as user×item and failed with `IndexError`; corrected to fit user×item.
- `export_arh_21-30-final.csv` contains week-31 rows and two extra show IDs; candidate schedule/catalog construction now filters exact target weeks 21–30.
- Initial association-rule source was slow because repeated membership checks used a list; changed catalog membership to a set. Its completed main run still lowered Recall@500 slightly versus expanded6, so it is excluded from current top source configuration.
- LightFM 1.17 required an isolated Python 3.12 build workaround: patch the temporary sdist's `__builtins__` handling and regenerate C extensions with Cython 0.29.37. The extension runs without OpenMP (single-thread warning). This is an environment workaround, not evidence of model quality.


## Result interpretation

The corrected A–E tripartite baseline has main strong-watch Recall@500 0.494101. Expanded sources reach 0.613961 equal-RRF, 0.626487 after early-only Recall@500 tuning, and full-union recall 0.887259. Candidate-only weights tuned by MAP@5 on early data reach main strong-target MAP@5 0.523572 vs 0.478560 equal-RRF, but reduce Recall@500 to 0.576003. Rolling expanded4 Recall@500 ranges 0.696–0.817 under equal RRF; frozen early weights improve those rolling scores.

The main Recall@500 remains below the working high-recall gate. No reference binary LightGBM, LGBMRanker, or new Kaggle submission has been run.

Next: decide whether to add another candidate source or relax the high-recall gate. Only after approval should reranker experiments begin.


## Later source ablations and frozen early weights

After creating this notebook draft, the main source set was extended with `airing_popularity`, then association-rule consequents. Weights for expanded5 and expanded6 were tuned on only the early history≤16→target17–20 split and evaluated frozen on main; expanded7 was retained only as an equal-RRF source ablation.

## Schedule-propensity candidate features

Expanded9 materializes user×channel, user weekday, user slot and exact user×channel×weekday×slot history factors. Future airing scores are aggregated per show with max, mean, sum and logsumexp; all four are written as candidate-table columns and used as candidate sources.


In [4]:
variants = [
    ("expanded5", "results/tables/exp5_candidate_recall_expanded5.csv", "results/tables/exp5_candidate_fusion_tuned_expanded5.csv"),
    ("expanded6", "results/tables/exp5_candidate_recall_expanded6.csv", "results/tables/exp5_candidate_fusion_tuned_expanded6.csv"),
    ("expanded7", "results/tables/exp5_candidate_recall_expanded7.csv", None),
]
for name, recall_path, tuned_path in variants:
    recall = pd.read_csv(recall_path)
    row500 = recall[(recall.threshold==0.8)&(recall.generator=="rrf_union")&(recall.cutoff==500)].iloc[0]
    full = recall[(recall.threshold==0.8)&(recall.generator=="rrf_union")].iloc[-1]
    print(f"{name} equal main: R@500={row500.recall:.6f}; full={full.recall:.6f}")
    if tuned_path:
        tuned = pd.read_csv(tuned_path)
        row = tuned[(tuned.split=="main_holdout")&(tuned.ranking=="early_tuned")&(tuned.threshold==0.8)&(tuned.cutoff==500)].iloc[0]
        print(f"{name} early-tuned main: R@500={row.recall:.6f}")


expanded5 equal main: R@500=0.609785; full=0.882983
expanded5 early-tuned main: R@500=0.615092
expanded6 equal main: R@500=0.613961; full=0.887259
expanded6 early-tuned main: R@500=0.626487
expanded7 association equal main: R@500=0.608301; full=0.887687


### Updated interpretation

The corrected A–E tripartite baseline has main strong-watch Recall@500 0.494101. Expanded sources reach 0.629229 equal-RRF (full union 0.891737), and early-only Recall@500 tuning gave 0.633883. Adding threshold-specific LightFM candidates gave equal-RRF 0.630009/full 0.896667 and early-tuned 0.636222. Candidate-only early MAP@5 tuning on expanded9 reached 0.568194 on main (equal RRF 0.479806), but Recall@500 was 0.600428.

The main Recall@500 remains below the user-confirmed 0.80 gate. No reference binary LightGBM, LGBMRanker, or new Kaggle submission has been run. Rolling windows include stronger subperiods but do not replace the primary split.

Next: improve candidate generation on the primary split; only after Recall@500 reaches 0.80 should reranker experiments begin.


In [5]:
import pandas as pd
d = pd.read_csv("results/tables/exp5_candidate_map_tuned.csv")
for threshold in (.5, .8):
    print(f"target threshold={threshold}")
    for ranking in ("equal_rrf", f"early_tuned_target{threshold:.1f}"):
        r = d[(d.split=="main_holdout") & (d.target_threshold==threshold) & (d.ranking==ranking)].iloc[0]
        print(f"{ranking}: MAP@5={r.MAP5:.6f}, Recall@500={r.Recall500:.6f}, users={int(r.n_users)}")


target threshold=0.5
equal_rrf: MAP@5=0.575607, Recall@500=0.586095, users=1244
early_tuned_target0.5: MAP@5=0.619004, Recall@500=0.531225, users=1244
target threshold=0.8
equal_rrf: MAP@5=0.478560, Recall@500=0.613961, users=1236
early_tuned_target0.8: MAP@5=0.523572, Recall@500=0.576003, users=1236


In [6]:
import pandas as pd
m=pd.read_csv("results/tables/exp5_candidate_map_tuned_widegrid.csv")
for ranking in ["equal_rrf","early_tuned_target0.8"]:
    r=m[(m.split=="main_holdout")&(m.target_threshold==.8)&(m.ranking==ranking)].iloc[0]
    print(f"main {ranking}: MAP@5={r.MAP5:.6f}, Recall@500={r.Recall500:.6f}, users={int(r.n_users)}")
w=pd.read_csv("results/tables/exp5_candidate_map_weights_widegrid.csv")
w=w[(w.target_threshold==.8)&(w.weight.isin([8,16]))]
print("weights at grid boundary:",dict(zip(w.source,w.weight)))


main equal_rrf: MAP@5=0.478560, Recall@500=0.613961, users=1236
main early_tuned_target0.8: MAP@5=0.524715, Recall@500=0.557062, users=1236
weights at grid boundary: {'repeat': 16.0, 'ppr': 8.0, 'airing_popularity': 8.0}


In [7]:
import pandas as pd
for name,path in [("main","results/tables/exp5_candidate_recall_expanded9.csv"),("early","results/tables/exp5_candidate_recall_expanded9_early.csv")]:
    d=pd.read_csv(path); r=d[(d.threshold==.8)&(d.generator=="rrf_union")]
    at500=r[r.cutoff==500].iloc[0]; full=r.iloc[-1]
    print(f"{name}: Recall@500={at500.recall:.6f} ({int(at500.hits)}/{int(at500.relevant)}), full={full.recall:.6f}")
c=pd.read_parquet("data/exp5_candidates_main_expanded9.parquet",columns=["user_id","tv_show_id","airing_propensity_max","airing_propensity_mean","airing_propensity_sum","airing_propensity_logsumexp"])
print("candidate rows/users:",len(c),c.user_id.nunique())
print("propensity nonnull:",{x:int(c[x].notna().sum()) for x in c.columns if x.startswith("airing_propensity_")})


main: Recall@500=0.629229 (25015/39755), full=0.891737
early: Recall@500=0.774756 (13645/17612), full=0.935612
candidate rows/users: 2605151 1244
propensity nonnull: {'airing_propensity_max': 621288, 'airing_propensity_mean': 621288, 'airing_propensity_sum': 621288, 'airing_propensity_logsumexp': 621288}


In [8]:
import pandas as pd
for name,path in [("propensity","results/tables/exp5_candidate_map_tuned_propensity.csv"),("propensity_widegrid","results/tables/exp5_candidate_map_tuned_propensity_widegrid.csv")]:
    d=pd.read_csv(path)
    r=d[(d.split=="main_holdout")&(d.target_threshold==.8)&(d.ranking=="early_tuned_target0.8")].iloc[0]
    e=d[(d.split=="main_holdout")&(d.target_threshold==.8)&(d.ranking=="equal_rrf")].iloc[0]
    print(f"{name}: main MAP@5 equal={e.MAP5:.6f}, early-tuned={r.MAP5:.6f}; Recall@500 tuned={r.Recall500:.6f}; users={int(r.n_users)}")


propensity: main MAP@5 equal=0.479806, early-tuned=0.556025; Recall@500 tuned=0.609860; users=1236
propensity_widegrid: main MAP@5 equal=0.479806, early-tuned=0.568194; Recall@500 tuned=0.600428; users=1236


## Additional collaborative-source check

RP3beta and EASE were added as extra candidates using the existing project formulas (`exp_grid_rp3_ease_ppr.py`): RP3beta binary α=0.5/β=0.3 and EASE binary λ=1000. The test did not improve candidate recall and the source pair was rejected.


In [9]:
import pandas as pd
for name,path in [("expanded9","results/tables/exp5_candidate_recall_expanded9.csv"),("expanded10 RP3/EASE","results/tables/exp5_candidate_recall_expanded10.csv")]:
    d=pd.read_csv(path); r=d[(d.threshold==.8)&(d.generator=="rrf_union")]
    top=r[r.cutoff==500].iloc[0]; full=r.iloc[-1]
    print(f"{name}: Recall@500={top.recall:.6f} ({int(top.hits)}/{int(top.relevant)}); full={full.recall:.6f}")


expanded9: Recall@500=0.629229 (25015/39755); full=0.891737
expanded10 RP3/EASE: Recall@500=0.604251 (24022/39755); full=0.887964


In [10]:
import pandas as pd
m=pd.read_csv("results/tables/exp5_candidate_recall_expanded11.csv")
r=m[(m.threshold==.8)&(m.generator=="rrf_union")]
a=r[r.cutoff==500].iloc[0]; f=r.iloc[-1]
t=pd.read_csv("results/tables/exp5_candidate_fusion_tuned_expanded11.csv")
x=t[(t.split=="main_holdout")&(t.ranking=="early_tuned")&(t.threshold==.8)&(t.cutoff==500)].iloc[0]
print(f"expanded11 equal main: Recall@500={a.recall:.6f} ({int(a.hits)}/{int(a.relevant)}); full={f.recall:.6f}")
print(f"expanded11 early-tuned main: Recall@500={x.recall:.6f} ({int(x.hits)}/{int(x.relevant)}); users={int(x.users)}")


expanded11 equal main: Recall@500=0.630009 (25046/39755); full=0.896667
expanded11 early-tuned main: Recall@500=0.636222 (25293/39755); users=1236
